# LLM Evaluation on Kaggle

Notebook này dùng riêng cho Kaggle Notebook để benchmark các LLM trên comparative quintuple extraction.

Trước khi chạy:
1. Add project folder dưới dạng Kaggle Input Dataset hoặc clone từ GitHub.
2. Thêm `OPENROUTER_API_KEY` trong `Settings -> Secrets`.
3. Chỉnh `PROJECT_INPUT_DIR`, `DATASETS`, `SPLIT`, `MODELS`, `PROMPT_STRATEGY` ở cell cấu hình.

## Nên dùng openrouter hay hf-local?

- Dùng `openrouter` khi:
  - Bạn muốn kết quả ổn định và không bị giới hạn bởi VRAM notebook.
  - Bạn cần chạy nhiều model API trong thời gian ngắn.
  - Bạn chấp nhận chi phí theo token.
- Dùng `hf-local` khi:
  - Bạn muốn tối ưu chi phí API và chấp nhận chi phí compute local.
  - Bạn chỉ benchmark model open-source từ Hugging Face.
  - Bạn muốn bật quantization (`HF_LOAD_IN_4BIT`) để tiết kiệm VRAM.

## Gợi ý VRAM cho hf-local (tham khảo)

- 3B model:
  - FP16/BF16: >= 8 GB VRAM
  - 4-bit: >= 4-6 GB VRAM
- 7B-8B model:
  - FP16/BF16: >= 16 GB VRAM
  - 4-bit: >= 8-12 GB VRAM
- 13B model:
  - FP16/BF16: >= 24 GB VRAM
  - 4-bit: >= 12-16 GB VRAM
- 30B+ model:
  - Thường cần multi-GPU hoặc không phù hợp với Kaggle GPU phổ biến.

Lưu ý:
- Kaggle thường phù hợp với `hf-local` + 4-bit cho model <= 7B/8B.
- Nếu báo OOM, ưu tiên đổi model nhỏ hơn trước.
- Hãy chạy `LIMIT > 0` để smoke test trước khi chạy full test set.

## Model gợi ý từ Hugging Face (EN + VI)

Shortlist cân bằng chất lượng/chi phí:
1. `Qwen/Qwen2.5-3B-Instruct`
2. `Qwen/Qwen2.5-7B-Instruct`
3. `mistralai/Mistral-7B-Instruct-v0.3`

Nếu có GPU tốt hơn, có thể mở rộng:
- `Qwen/Qwen2.5-14B-Instruct`
- `meta-llama/Llama-3.1-8B-Instruct`

## Benchmark matrix 3 x 3 đề xuất (so sánh E-T5-MACRO-F1)

- Models:
  - `Qwen/Qwen2.5-3B-Instruct`
  - `Qwen/Qwen2.5-7B-Instruct`
  - `mistralai/Mistral-7B-Instruct-v0.3`
- Strategies:
  - `zero-shot`
  - `few-shot`
  - `cot`

Tổng cộng 9 run:
1. 3B x zero-shot
2. 3B x few-shot
3. 3B x cot
4. 7B x zero-shot
5. 7B x few-shot
6. 7B x cot
7. Mistral-7B x zero-shot
8. Mistral-7B x few-shot
9. Mistral-7B x cot

In [ ]:
# Cell 1 · Kaggle paths
import os

assert os.path.exists('/kaggle/working'), 'Notebook này chỉ dành cho Kaggle.'

PROJECT_INPUT_DIR = '/kaggle/input/msc-project'
WORK_DIR = '/kaggle/working/msc-project'
LLMEVAL_DIR = os.path.join(WORK_DIR, 'llm_eval')
DATASETS_ROOT = os.path.join(WORK_DIR, 'datasets')
OUTPUT_DIR = os.path.join(LLMEVAL_DIR, 'results')
CACHE_DIR = os.path.join(LLMEVAL_DIR, 'cache')

print('Input project dir:', PROJECT_INPUT_DIR)
print('Working project dir:', WORK_DIR)

In [ ]:
# Cell 2 · Prepare project in /kaggle/working
GITHUB_REPO = ''  # tùy chọn: clone nếu không dùng Kaggle Input

import os
import shutil
import subprocess

if not os.path.exists(LLMEVAL_DIR):
    if os.path.exists(PROJECT_INPUT_DIR):
        shutil.copytree(PROJECT_INPUT_DIR, WORK_DIR, dirs_exist_ok=True)
        print('Copied project from Kaggle Input Dataset.')
    elif GITHUB_REPO:
        subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, WORK_DIR], check=True)
        print('Cloned project from GitHub.')
    else:
        raise FileNotFoundError('Không tìm thấy project trong Kaggle Input và GITHUB_REPO đang để trống.')
else:
    print(f'Project already exists at {WORK_DIR}')

In [ ]:
# Cell 3 · Install dependencies
import os
import subprocess
import sys

reqs = os.path.join(LLMEVAL_DIR, 'requirements.txt')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', reqs, '-q'], check=True)
print('Dependencies installed.')

In [ ]:
# Cell 4 · Load OpenRouter API key from Kaggle Secrets
import os
from kaggle_secrets import UserSecretsClient

os.environ['OPENROUTER_API_KEY'] = UserSecretsClient().get_secret('OPENROUTER_API_KEY')
assert os.environ.get('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY not found in Kaggle Secrets.'
print('API key loaded.')

In [ ]:
# Cell 5 · Configuration
DATASETS = 'camera-coqe,vcom-data'
SPLIT = 'test'
PROMPT_STRATEGY = 'few-shot'  # zero-shot | few-shot | cot

# Provider options: openrouter | hf-local
PROVIDER = 'openrouter'
HF_DTYPE = 'auto'             # auto | float16 | bfloat16
HF_LOAD_IN_4BIT = False

MODELS = [
    'openai/gpt-4o-mini',
    'anthropic/claude-3.5-haiku',
    'google/gemini-2.0-flash-001',
    'deepseek/deepseek-chat',
    'qwen/qwen-2.5-72b-instruct',
    'meta-llama/llama-3.3-70b-instruct',
]

TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 256
SLEEP_SECONDS = 0.3
LIMIT = 0

print('Configuration ready.')

In [ ]:
# Cell 6 · Run evaluation
import os
import subprocess
import sys

cmd = [
    sys.executable, os.path.join(LLMEVAL_DIR, 'run_eval.py'),
    '--datasets', DATASETS,
    '--split', SPLIT,
    '--models', *MODELS,
    '--provider', PROVIDER,
    '--prompt-strategy', PROMPT_STRATEGY,
    '--temperature', str(TEMPERATURE),
    '--max-output-tokens', str(MAX_OUTPUT_TOKENS),
    '--sleep-seconds', str(SLEEP_SECONDS),
    '--datasets-root', DATASETS_ROOT,
    '--output-dir', OUTPUT_DIR,
    '--cache-dir', CACHE_DIR,
]

if PROVIDER == 'openrouter':
    cmd += [
        '--base-url', 'https://openrouter.ai/api/v1',
        '--api-key-env', 'OPENROUTER_API_KEY',
    ]
elif PROVIDER == 'hf-local':
    cmd += [
        '--hf-dtype', HF_DTYPE,
    ]
    if HF_LOAD_IN_4BIT:
        cmd += ['--hf-load-in-4bit']
else:
    raise ValueError(f'Unsupported PROVIDER: {PROVIDER}')

if LIMIT > 0:
    cmd += ['--limit', str(LIMIT)]

print('Running command:')
print(' '.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=False)
print('Exit code:', result.returncode)

In [ ]:
# Cell 7 · Show summary
import json
import pathlib

summary_file = pathlib.Path(OUTPUT_DIR) / f'summary__{SPLIT}.json'

if summary_file.exists():
    with open(summary_file, 'r', encoding='utf-8') as f:
        rows = json.load(f)
    try:
        import pandas as pd
        df = pd.DataFrame([
            {
                'dataset': r['dataset'],
                'model': r['model'],
                'E-T5-MACRO-F1': round(r.get('E-T5-MACRO-F1', 0), 4),
                'E-T4-F1': round(r.get('E-T4-F1', 0), 4),
                'E-CEE-MICRO-F1': round(r.get('E-CEE-MICRO-F1', 0), 4),
            }
            for r in rows
        ]).sort_values(['dataset', 'E-T5-MACRO-F1'], ascending=[True, False])
        print(df.to_string(index=False))
    except ImportError:
        print(rows)
else:
    print('Summary file not found.')

In [ ]:
# Cell 8 · Results location
import pathlib

results_path = pathlib.Path(OUTPUT_DIR)
print('Results saved at:', results_path)
print('Use the right-side Output/Data panel in Kaggle to download files if needed.')